In [1]:
import os
os.chdir("../")

In [2]:
from dataclasses import dataclass
from pathlib import Path
from box import ConfigBox

@dataclass
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: Path  # краще Path для гнучкості
    unzip_data: Path
    schema: dict       # виправлено назву відповідно до ініціалізації

In [3]:
from src.ds_edep.constants import *
from src.ds_edep.utils.common import read_yaml, create_directories

[2026-06-06 11:45:58,685: INFO: __init__: Logger setup completed!]


In [4]:
class ConfigManager:
    def __init__(self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):
        
        # Виправлено назви атрибутів (прибираємо _filepath, бо це вже об'єкти ConfigBox)
        self.params = read_yaml(params_filepath)
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS
        
        create_directories([config.root_dir])
        
        class_conf = DataValidationConfig(
            root_dir = Path(config.root_dir),
            STATUS_FILE = Path(config.STATUS_FILE),
            unzip_data = Path(config.unzip_data_dir),
            schema = schema  # тепер збігається з назвою в датакласі
        )
        
        return class_conf

In [5]:
import pandas as pd

In [6]:
class DataValidation:
    # Виправлено тип анотації на ваш датаклас
    def __init__(self, config: DataValidationConfig):
        self.config = config
    
    def validate_all_column(self) -> bool:
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data)
            
            # Перетворюємо типи кожної колонки на звичайні РЯДКИ (str), 
            # щоб їх можна було порівняти з рядками з вашого schema.yaml
            current_schema = {col: str(dtype) for col, dtype in data.dtypes.to_dict().items()}
            expected_schema = self.config.schema
            
            # Порівнюємо два словники
            if current_schema != expected_schema:
                validation_status = False
            
            # Записуємо статус у файл
            with open(self.config.STATUS_FILE, 'w', encoding="utf-8") as f:
                f.write(f"Validation status: {validation_status}")
                
            return validation_status
            
        except Exception as e:
            raise e

In [7]:
try:
    config = ConfigManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_column()
except Exception as e:
    raise e

[2026-06-06 11:45:59,332: INFO: common: created directory at: artifacts/data_validation]
